# Intent classification with BERT
This notebook trains a multilingual BERT classifier to recognize user intents from text.

We will:
1. Load the CSV dataset.
2. Map text labels to numeric IDs.
3. Split the data into training, validation, and test sets.
4. Tokenize the text for a transformer model.
5. Load a pretrained model and fine-tune it.
6. Train using the Hugging Face Trainer API.

Each code cell is explained with the theory behind what it does.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,AutoModelForMaskedLM
from datasets import Dataset
import torch

c:\Users\hp\chatbot_health\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Imports and libraries
- `pandas` handles tabular data from CSV files.
- `sklearn.model_selection.train_test_split` splits data into train, validation, and test sets.
- `transformers` provides pretrained models and tokenizers from Hugging Face.
- `datasets.Dataset` converts pandas data into a format suitable for transformer training.
- `torch` is the underlying deep learning framework used by the model.

In [4]:
df=pd.read_csv(r'C:\Users\hp\chatbot_health\backend\datasets\intent\dataset_f.csv')

## Dataset loading from CSV
Load the CSV file containing the intent dataset into a pandas DataFrame.
Each row is one example with text and an intent label.

This is the raw data source for training and evaluation.

In [5]:
labels = df["intent"].unique().tolist()
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["intent"].map(label2id)
print(label2id)
print(id2label)


{'demande_urgence': 0, 'remboursement_mutuelle': 1, 'description_symptome': 2, 'demande_posologie': 3, 'demande_consultation': 4, 'demande_medicament': 5, 'demande_prix': 6}
{0: 'demande_urgence', 1: 'remboursement_mutuelle', 2: 'description_symptome', 3: 'demande_posologie', 4: 'demande_consultation', 5: 'demande_medicament', 6: 'demande_prix'}


## Label encoding
Machine learning models require numeric labels, not text strings.
This cell builds two dictionaries:
- `label2id`: map string labels to numbers
- `id2label`: map numbers back to strings

Then it applies the numeric label to the dataset.

In [6]:
train, temp = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

## Train/validation/test split
Split the dataset into three parts:
- training set: used to teach the model
- validation set: used to tune hyperparameters and check for overfitting during training
- test set: used once at the end to evaluate final performance

Using `random_state` ensures the split is reproducible.

In [7]:
print(f"Train size: {len(train)}, Validation size: {len(val)}, Test size: {len(test)}")

Train size: 734, Validation size: 157, Test size: 158


## Data size check
Print the size of each subset to confirm the split proportion.
This helps verify there is enough data for training and evaluation.

In [20]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

## Tokenizer loading
Load a pretrained tokenizer for atlasia xlm roberta model .
A tokenizer converts raw text into token IDs and attention masks, which are the numerical inputs the model understands.
This tokenizer supports many languages, so it can handle both Roman-script text and Arabic text in the same dataset.

Note: the tokenizer does not train on your data; it only converts your text into the format required by the model.

In [21]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train[["text", "label"]])
val_dataset = Dataset.from_pandas(val[["text", "label"]])
test_dataset = Dataset.from_pandas(test[["text", "label"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 158/158 [00:00<00:00, 2699.70 examples/s]


## Tokenization and dataset conversion
This cell:
1. Defines a `tokenize` function that converts raw text to token IDs and attention masks.
2. Creates Hugging Face `Dataset` objects from pandas data.
3. Applies the tokenizer to each split with batching.

The resulting datasets are ready to feed into the transformer model.

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 860.96it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

## Load pretrained model for classification
Load a multilingual BERT model with a sequence classification head.
The `num_labels` argument tells the model how many intent classes to predict.
Providing `id2label` and `label2id` ensures output labels are human-readable.

In [23]:
import accelerate
print(accelerate.__version__)

1.13.0


## Accelerator version check
The `accelerate` library helps manage hardware and distributed training.
Here we simply print the version to confirm it is installed and available.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no", 
    load_best_model_at_end=False,
    logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Training configuration
Define the training behavior with Hugging Face `TrainingArguments`.
- `output_dir`: folder for saved checkpoints
- `num_train_epochs`: how many passes over the training data
- `per_device_train_batch_size`: batch size per GPU/CPU device
- `eval_strategy` and `save_strategy`: how often to evaluate and save
- `load_best_model_at_end`: keep the best validation model

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

## Trainer setup
Create the `Trainer` object which orchestrates training, evaluation, and optimization.
It takes the model, training arguments, and datasets, and handles the training loop internally.

In [26]:
trainer.train()

c:\Users\hp\chatbot_health\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,1.693015
2,No log,1.214464
3,No log,0.809964
4,No log,0.500813
5,No log,0.418728


TrainOutput(global_step=230, training_loss=1.055509617017663, metrics={'train_runtime': 1849.0662, 'train_samples_per_second': 1.985, 'train_steps_per_second': 0.124, 'total_flos': 54695638195500.0, 'train_loss': 1.055509617017663, 'epoch': 5.0})

## Start fine-tuning
Run the training process.
The `Trainer` will iterate over the training data, compute loss, update model weights, and evaluate on the validation set at each epoch.

## Save the trained model
Save the fine-tuned model and tokenizer to disk so they can be reused later without retraining.
The saved folder can be loaded again with `AutoModelForSequenceClassification.from_pretrained` and `AutoTokenizer.from_pretrained`.

In [58]:
model.save_pretrained("../models/intent_model/")
tokenizer.save_pretrained("../models/intent_model/tokenizer/")

Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.18s/it]


('../models/intent_model/tokenizer/tokenizer_config.json',
 '../models/intent_model/tokenizer/tokenizer.json')

## Evaluate on the test set
Use the trained model to make predictions on the unseen test data.
This step measures how well the model generalizes to new examples.

In [31]:
predictions = trainer.predict(test_dataset)
print(predictions.metrics)

c:\Users\hp\chatbot_health\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.5639232993125916, 'test_runtime': 13.5792, 'test_samples_per_second': 11.635, 'test_steps_per_second': 0.736}


## Generate classification metrics
Convert raw model scores into predicted labels and compare them to the true labels.
Print a classification report showing precision, recall, and F1-score for each intent class.

In [44]:
import numpy as np
from sklearn.metrics import classification_report

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=list(label2id.keys())))

                        precision    recall  f1-score   support

       demande_urgence       0.91      0.83      0.87        24
remboursement_mutuelle       0.80      0.84      0.82        19
  description_symptome       0.92      0.88      0.90        25
     demande_posologie       0.83      0.76      0.79        25
  demande_consultation       0.89      0.96      0.92        25
    demande_medicament       0.83      0.87      0.85        23
          demande_prix       0.67      0.71      0.69        17

              accuracy                           0.84       158
             macro avg       0.83      0.84      0.83       158
          weighted avg       0.84      0.84      0.84       158



## Real-world inference examples
Load the fine-tuned model into a text classification pipeline and run it on example sentences.
This demonstrates how the trained intent classifier can be used in a real application.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

phrases = [
    
   
    
      
    

]

for phrase in phrases:
    result = classifier(phrase)
    print(f"{phrase} → {result[0]['label']} ({result[0]['score']:.2f})")

I have a headache and fever. → demande_consultation (0.43)
What are the symptoms of diabetes? → demande_consultation (0.43)
How can I improve my sleep quality? → demande_consultation (0.46)
I need advice on managing stress. → demande_consultation (0.50)
What should I do if I have a sore throat? → remboursement_mutuelle (0.21)
